Grid File IO
============

We have tools for handling data stored on "the grid", i.e. the `/sptgrid` directory accessible on `scott` and `amundsen`.  While the directory is mounted over the network (via NFS) for read-only access, it uses a significant amount of network bandwidth over a relatively constrained path to read files directly that way.  We thus prefer that people use the GFAL tool suite to read data -- `gfal-ls`, `gfal-copy`, etc.  To make it easier to handle data this way, we have added some convenient Python tools for reading and writing files from `scott` and `amundsen`.

To get started here, make sure to create a fresh grid token, using `token-init` on the command line.  Check that your token is active using `token-info`.

The grid tools in the `spt3g.cluster` package use a global `GridCacher` instance to keep track of what files have been copied to local disk, and should be cleared out automatically once your Python process exits.  You do not need to access the cacher instance directly, but it may be useful for debugging.

In [ ]:
from spt3g import core, cluster
cacher = cluster.GridCacher()

You can also check that your grid token is working using the `get_grid_token()` function, which will raise an error if your toekn is out of date:

In [ ]:
cluster.get_grid_token()

For reading, G3 files, we have grid-aware versions of the `G3Reader` and `G3Writer` pipeline modules, and the `G3File` function:

In [ ]:
pipe = core.G3Pipeline()
pipe.Add(cluster.GridReader, filename="/sptgrid/user/arahlin/test.g3")
pipe.Add(core.Dump)
pipe.Add(cluster.GridWriter, filename="/sptgrid/user/arahlin/test2.g3")
pipe.Run()

Let's take a look at what's in the cache, and then clean up the files.

In [ ]:
print("Cached files:", cacher.local_files)
print("Disk usage (GB)", cacher.used)
cacher.cleanup()
print("After cleanup:", cacher.local_files)

Here's an example using the `GridFile` function, equivalent to `G3File`:

In [ ]:
for fr in cluster.GridFile("/sptgrid/user/arahlin/test2.g3"):
    print(fr)

For non-G3 files, we have two functions for reading and writing data.  To load a file from the grid, you need access to the local filename, so the `get_grid_file` function returns the path in the cache directory.

In [ ]:
local_filename = cluster.get_grid_file("/sptgrid/user/arahlin/test.txt")
print("Cached path:", local_filename)
print("Contents:", open(local_filename, "r").read())

To write files to the grid, we need access to the cached path before writing to it, and then upload the finished file to the grid.  To do this, we use a context manager, `put_grid_file`.  This function yields a filename to the block within the the `with` statement, then uploades that file to its grid location once that block completes successfully.  For example:

In [ ]:
with cluster.put_grid_file("/sptgrid/user/arahlin/test.txt", overwrite=True) as local_filename:
    print(local_filename)
    with open(local_filename, "w") as f:
        f.write("testing\n")